In [10]:
from mlflow.tracking import MlflowClient

mlflow_tracking = "sqlite:///mlflow.db"

client = MlflowClient(tracking_uri=mlflow_tracking)


/Users/macbook/Projects/mlops-1/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [ ]:
client.create_experiment( name ="trying out experiments")


'2'

In [11]:
client.get_registered_model("nyc-taxi-data")

<RegisteredModel: aliases={}, creation_timestamp=1731082507919, description='', last_updated_timestamp=1731180407669, latest_versions=[<ModelVersion: aliases=[], creation_timestamp=1731082508001, current_stage='Staging', description='', last_updated_timestamp=1731083360148, name='nyc-taxi-data', run_id='5e4ecc9c718b4a6abd1e9c490f4d1428', run_link='', source='/Users/macbook/Projects/mlops-1/02-experiment-tracking/mlruns/1/5e4ecc9c718b4a6abd1e9c490f4d1428/artifacts/gradient_boosting_model', status='READY', status_message=None, tags={}, user_id=None, version=1>,
 <ModelVersion: aliases=[], creation_timestamp=1731083255182, current_stage='Archived', description='', last_updated_timestamp=1731083319473, name='nyc-taxi-data', run_id='02b97cc1f3bb4d4382492c0d3dccd281', run_link='', source='models:/nyc-taxi-data/3', status='READY', status_message=None, tags={}, user_id=None, version=4>,
 <ModelVersion: aliases=[], creation_timestamp=1731082564867, current_stage='Production', description='', la

In [15]:
model_name = 'nyc-taxi-data'
model_version = client.get_latest_versions(model_name)

for version in model_version:
    print(f"version: {version.version}, stage: {version.current_stage}")     

version: 1, stage: Staging
version: 4, stage: Archived
version: 3, stage: Production


/var/folders/58/1jrwwbcj7mb8n97wtbklvw280000gn/T/ipykernel_25231/203958764.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  model_version = client.get_latest_versions(model_name)


In [13]:
run_id = "5e4ecc9c718b4a6abd1e9c490f4d1428"

In [29]:
model_version = "2"
new_stage = "Production"   

client.update_model_version(
    name = model_name,
    version =model_version,
    description = "The model version {model_version} is now in stage {new_stage}",
    
    
)

<ModelVersion: aliases=[], creation_timestamp=1731082537068, current_stage='Archived', description='The model version {model_version} is now in stage {new_stage}', last_updated_timestamp=1731175619972, name='nyc-taxi-data', run_id='7f746365f24146ed8580a828751252f4', run_link='', source='/Users/macbook/Projects/mlops-1/02-experiment-tracking/mlruns/1/7f746365f24146ed8580a828751252f4/artifacts/random_forest_model', status='READY', status_message=None, tags={}, user_id=None, version=2>

In [17]:
import pandas as pd
from sklearn.metrics import mean_squared_error


def read_dataframe(path):
    df = pd.read_parquet(path)
    
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)
    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    
    df["duration"] = (df.lpep_dropoff_datetime - df.lpep_pickup_datetime)
    df.duration = df.duration.apply(lambda x: x.total_seconds()/60)
    
    df = df[(df.duration >= 1) & (df.duration <= 60)]    
    
    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype('str')
    
    return df
def preprocess(df):
    df['PU_DO'] = df.PULocationID + "_" + df.DOLocationID
    categoical = ['PU_DO']
    numerical = ['trip_distance']
    train_dicts = df[categoical + numerical].to_dict(orient='records')
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(train_dicts)

    return X_train

def test_model(name, stage, X_test, y_test):
    model = mlflow.pyfunc.load_model(f"models:/{name}/{stage}")
    y_pred = model.predict(X_test)
    return {"rmse": mean_squared_error(y_test, y_pred, squared=False)}    
                 
    
      

In [18]:
df = read_dataframe("data/green_tripdata_2021-03.parquet")

In [19]:
X_test = preprocess(df)

In [20]:
target = "duration"
y_test = df[target].values


In [24]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")

In [ ]:
test_model(model_name, "Production", X_test, y_test)